<a href="https://colab.research.google.com/github/Durogene/tp-information-retrieval-with-llm-student-version/blob/main/1-Recherche%20d'information%20classique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 1. - Recherche d'Information classique

Dans cette partie, nous allons mettre en œuvre des principes et modèles classiques de Recherche d'Information. Le jeu de données est une collection de livres au format texte (.txt) de Henry Rider Haggard. Jetez un œil à ces documents dans le dossier _data_.

En sortie de ce module, vous serez capable de :

- Construire un index inversé ;
- Effectuer des requêtes simples selon le modèle booléen :
- Calculer la pondération des termes selon la méthode TF-IDF ;
- Mettre en œuvre le modèle vectoriel de recherche d'information et y appliquer des requêtes.

### Import des bibliothèques logicielles et configuration

Les lignes suivantes permettent d'instancier un objet de la classe `IRSystem` représentant notre moteur de recherche et de charger les données en RAM.

In [2]:
import sys

# Vérifie si le code est exécuté sur Google Colab
if 'google.colab' in sys.modules:
    # Commandes à exécuter uniquement sur Google Colab
    !git clone https://github.com/Durogene/tp-information-retrieval-with-llm-student-version.git
    %cd tp-information-retrieval-with-llm-student-version
else:
    # Commandes à exécuter si ce n'est pas sur Google Colab
    print("Pas sur Google Colab, ces commandes ne seront pas exécutées.")

Cloning into 'tp-information-retrieval-with-llm-student-version'...
remote: Enumerating objects: 2265, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 2265 (delta 32), reused 34 (delta 17), pack-reused 2212 (from 1)
Receiving objects: 100% (2265/2265), 13.93 MiB | 18.54 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/tp-information-retrieval-with-llm-student-version


#### Chargement des données

Les lignes ci-dessous permettent de charger les données qui sont un ensemble de 60 livres au format texte (.txt) d'[Henry Rider Haggard](https://fr.wikipedia.org/wiki/Henry_Rider_Haggard).


In [3]:
from classic_ir.IRSystem import *

# !rm -rf ./data/RiderHaggard/stemmed
ir_system = IRSystem()
ir_system.read_data('./data/RiderHaggard') # chargement des données et prétraitement des documents (stemming).

Reading in documents...
Stemming Documents...
Cetywayo and his White Neighbours 0.txt
    Doc 1 of 60: Cetywayo and his White Neighbours


/content/tp-information-retrieval-with-llm-student-version/classic_ir/IRSystem.py:35: SyntaxWarning: invalid escape sequence '\d'
  title_pattern = re.compile('(.*) \d+\.txt')


The World's Desire 2763.txt
    Doc 2 of 60: The World's Desire
Queen of the Dawn (1925) 0200381.txt
    Doc 3 of 60: Queen of the Dawn (1925)
Maiwa's Revenge 2713.txt
    Doc 4 of 60: Maiwa's Revenge
The Wizard 2893.txt
    Doc 5 of 60: The Wizard
Marie An Episode in The Life of the late Allan Quatermain 1690.txt
    Doc 6 of 60: Marie An Episode in The Life of the late Allan Quatermain
The Ghost Kings 8184.txt
    Doc 7 of 60: The Ghost Kings
Wisdom's Daughter (1923) 0200181.txt
    Doc 8 of 60: Wisdom's Daughter (1923)
Cleopatra 2769.txt
    Doc 9 of 60: Cleopatra
Love Eternal 3709.txt
    Doc 10 of 60: Love Eternal
Red Eve 3094.txt
    Doc 11 of 60: Red Eve
Jess 5898.txt
    Doc 12 of 60: Jess
Nada the Lily 1207.txt
    Doc 13 of 60: Nada the Lily
The Mahatma and the Hare 2764.txt
    Doc 14 of 60: The Mahatma and the Hare
Lysbeth, a Tale of the Dutch 5754.txt
    Doc 15 of 60: Lysbeth, a Tale of the Dutch
Allan and the Ice Gods (1927) 0200201.txt
    Doc 16 of 60: Allan and the Ic

### Exercice 1. - Construction de l'index inversé

Ce premier exercice a pour objectif de construire l'index inversé non positionnel. L'attribut `self.inverted_index` est un tableau associatif qui associe une liste d'entiers (docIDs) à un mot (word).

Documentation ici https://docs.python.org/3/library/collections.html#collections.defaultdict.

Exercice : modifier la fonction `index` pour calculer l'index inversé.

**Indication** : profitez du parcours des documents pour remplir aussi `self.tf`, qui associe à chaque docID un `Counter` donnant le nombre d'occurrences de chaque mot dans ce document. Vous en aurez besoin à l'exercice 3 pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [4]:
from typing_extensions import Counter
# Exercice 1. Indexation

def index(self):
    """
    Construit l'index inversé et le stocke dans self.inverted_index.

    Dans le code ci-dessous, seul le dictionnaire des tokens est construit. Les postings lists sont vides.
    """
    print("Indexing...")
    self.tf = defaultdict(Counter) # tf est un dictionnaire qui à un document associe un Counter de mots (à remplir, utilisé à l'exercice 3).
    inverted_index = defaultdict(list) # inverted_index est un dictionnaire qui à un mot associe une liste de documents.
    for word in self.vocab:
        inverted_index[word] = list()
    for doc_id, doc in enumerate(self.docs):
            self.tf[doc_id]=Counter(doc)
            for word in self.tf[doc_id]:
               inverted_index[word].append(doc_id)

    self.inverted_index = inverted_index

# Ne pas modifier les lignes ci-dessous
IRSystem.index = index
ir_system.index()
run_tests(ir_system, part=0)

Indexing...
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 2. - Recherche booléenne

Ce deuxième exercice a pour objectif d'implémenter la recherche booléenne. La requête `query` est une liste de mots _racinisés_ (stemming de Porter) et l'algorithme doit rechercher les documents qui contiennent TOUS ces mots.


Exercice : modifier la fonction `boolean_retrieve` pour implémenter la recherche booléenne.


Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [5]:
# Exercice 2. Recherche booléenne
def boolean_retrieve(self, query):

    """
    A partir d'une requête sous la forme d'une liste de mots *racinisés*,
    retourne la liste des documents dans lesquels *tous* ces mots apparaissent (ie une requête AND).
    Retourne une liste vide si la requête ne retourne aucun document.

    Dans le code ci-dessous, tous les documents répondent.
    """

    docs = list()
    if not query:
        return docs
    docs = set(self.get_posting(query[0]))
    for word in query[1:]:
        docs = docs.intersection(self.get_posting(word))
    return sorted(docs)


# Ne pas modifier les lignes ci-dessous
IRSystem.boolean_retrieve = boolean_retrieve
run_tests(ir_system, part=1)

===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 3. - Calcul des poids TF-IDF des termes dans les documents

Dans ce troisième exercice, l'objectif est de pré-calculer les poids TF-IDF pour chaque terme dans chaque document. Pour ce faire, appliquer la formule vue en cours. Utiliser le logarithme en base 10.


Exercice : modifier la fonction `compute_tfidf` pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [6]:
# Exercice 3. calcul des scores tf-idf
def compute_tfidf(self):
    """
    Calcule les scores tf-idf pour tous les mots de tous les documents et les stocke dans self.tfidf.

    Dans le code ci-dessous, les scores tf-idf sont tous nuls.
    """
    print("Calculating tf-idf...")

    self.tfidf = defaultdict(Counter)
    N = len(self.docs)  # N = nombre de documents
    for word in self.vocab:
       df = len(self.get_posting(word))
       idf = math.log10(N / df) if df > 0 else 0.
       for i in range(N):
            try:
                tf = self.tf[i][word]
                self.tfidf[i][word] = (1 + math.log10(tf)) * idf if tf > 0 else 0.
            except ValueError:
                self.tfidf[i][word] = 0.

# Ne pas modifier les lignes ci-dessous
IRSystem.compute_tfidf = compute_tfidf
ir_system.compute_tfidf()
run_tests(ir_system, part=2)

Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


#### Question - Du bruit dans le corpus ?

Les livres proviennent du [Projet Gutenberg](https://www.gutenberg.org/). Ouvrez quelques fichiers du dossier `data/RiderHaggard/raw` et regardez leur dernière ligne : la plupart se terminent par une mention ajoutée par le Projet Gutenberg, du type « End of the Project Gutenberg EBook of She, by H. Rider Haggard ».

1. Affichez, pour les termes `gutenberg`, `ebook` et `project`, le nombre de documents qui les contiennent (df) et leur idf. Rappel : `self.inverted_index` attend des mots racinisés, utilisez `get_posting_unstemmed`.
2. Pourquoi le poids TF-IDF de `gutenberg` reste-t-il faible dans tous les documents, alors que ce mot n'a rien à voir avec les romans ?
3. Quel serait l'idf de `gutenberg` si un seul livre contenait cette mention ? Quel risque cela poserait-il pour la recherche ?
4. Quel traitement proposeriez-vous avant l'indexation ? Faites le lien avec la partie du cours « Représenter les documents ».

In [7]:
# Question - Du bruit dans le corpus ?
N = len(ir_system.docs)
for word in ['gutenberg', 'ebook', 'project']:
    # A COMPLETER : afficher df et idf du mot
    df = len(ir_system.get_posting_unstemmed(word))
    idf = math.log10(N / df) if df > 0 else 0

    print("Mot :", word)
    print("DF :", df)
    print("IDF :", idf)


# Vos réponses aux questions 2 à 4 :
# 2.Vu que le mot gutenberg apparait dans les mentions ajoutées à la fin de pas mal de documents
#son df est donc eleve ce qui entraine un idf faible
# 3.Si un seul livre contient le mot « gutenberg », son IDF serait de log10(60/1) à  peu près 1,778
# le risque c'est que les résultat peuvent etre fausser car le moteur de recherche pourrait donné trop de poids à un livre
#même sans aucun rapport avec la recherche
# 4.Avant l'indexation on pourrait par exemple supprimer les mentions , les informations qui ne font pas partie
#ou qui ne representent pas vraiment le contenu du document


Mot : gutenberg
DF : 52
IDF : 0.062147906748844434
Mot : ebook
DF : 38
IDF : 0.19836765376683346
Mot : project
DF : 60
IDF : 0.0


### Exercice 4. - Calcul de la similarité cosinus.

Dans ce quatrième exercice, l'objectif est de calculer la similarité cosinus entre le vecteur requête `query` et chaque vecteur document. Pour ce faire, appliquer la formule vue en cours en considérant les éléments suivants :
- **Vecteurs documents** : poids TF-IDF calculés à l'exercice 3, **normalisés L2** (on divise par la norme du vecteur document).
- **Vecteur requête** : poids TF logarithmique `1 + log10(tf)`, **sans IDF et sans normalisation**.
- Utiliser le logarithme en base 10.

Le score attendu est donc :

$$score(q, d) = \frac{\sum_{t \in q} w_{t,q} \cdot w_{t,d}}{\lVert d \rVert_2}$$

Ne pas normaliser le vecteur requête ne change pas le classement (la norme de la requête est la même pour tous les documents), mais change la valeur du score, qui est vérifiée par le test. En notation SMART, il s'agit du schéma **ltc.lnn**.

Exercice : modifier la fonction `rank_retrieve` pour implémenter la similarité cosinus.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [8]:
# Exercice 4. Similarité cosinus
def rank_retrieve(self, query):
    """
    A partir d'une requête (une liste de mots), retourne la liste des 10 meilleurs documents (docID, score), classés par score décroissant, en appliquant la similarité cosinus.

    Dans l'exemple ci-dessous. C'est la mesure de Jaccard qui est utilisée.
    """
    scores = [0.0 for _ in range(len(self.docs))]

    query_tf = Counter(query)

    query_weights = {}
    for word, tf in query_tf.items():
        query_weights[word] = 1 + math.log10(tf)

    for d in range(len(self.docs)):
        dot_product = 0.0

        for word, query_weight in query_weights.items():
            document_weight = self.tfidf[d][word]
            dot_product += query_weight * document_weight

        document_norm = math.sqrt(
            sum(weight ** 2 for weight in self.tfidf[d].values())
        )

        scores[d] = dot_product / document_norm if document_norm != 0 else 0.0

    # Tri des scores
    ranking = [idx for idx, sim in sorted(enumerate(scores),
                                        key=lambda pair: pair[1], reverse=True)]
    results = []
    for i in range(10):
        results.append((ranking[i], scores[ranking[i]]))
    return results

# Ne pas modifier les lignes ci-dessous
IRSystem.rank_retrieve = rank_retrieve
run_tests(ir_system, part=3)

===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
